# Traceprop-LLM — Table 2 (head-to-head) + tracked-parameters sweep rerun

**Why this notebook is different:** the previous notebook (`exp25_llm_inline_overhead_colab.ipynb`) hit the same bug twice — Colab's "save to GitHub" writes back whatever cell *source* is loaded in your browser tab, so a tab opened before a fix was pushed silently reverts the fix when it auto-saves. Both times, a `--repeats 5` run got reported and used as if it were the fixed `--repeats 20` run.

This notebook has exactly **one substantive cell**: it calls `exp34_table2_and_sweep.py`, which runs all of Table 2 (GPT-2, Pythia-410M, Pythia-1B) plus the tracked-parameters sweep as **one Python process**, not per-cell `!python` invocations. There's no cell state to go stale — whatever's in the repo when you run it is what runs. It's also resumable: each job's result is saved to disk immediately (and backed up to Drive), and a rerun skips any job that already has a result file.

**Must run on the L4** (Runtime → Change runtime type → GPU → L4). Table 1 was measured on an L4; Kaggle only offers T4/P100, and mixing GPUs across Table 1 and Table 2 makes the percentages and post-hoc seconds incomparable. Do not run this on Kaggle.

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Pin to known-good versions -- an unpinned transformers install can pull a
# currently-broken release (confirmed locally: NameError in
# transformers/integrations/accelerate.py on an unpinned install).
!pip -q install "transformers==4.44.2" "peft==0.13.2" "accelerate==0.34.2"
!pip -q uninstall -y torchao 2>/dev/null
import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert 'L4' in torch.cuda.get_device_name(0), "This must run on an L4 -- got a different GPU. Change runtime type before continuing."

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/traceprop_runs', exist_ok=True)

In [ ]:
import getpass
TOKEN = getpass.getpass('GitHub token: ').strip()
url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git'
!git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
%cd /content/Traceprop
!pip -q install -e .

## Run everything (one process, ~40-60 min)

Table 2's three models take roughly the same order of time as the original notebook's runs did; the sweep adds 4 more full timing blocks at Table 1's config (batch=16, seq=64, 20 repeats each). Budget an hour.

**If the runtime disconnects partway through:** just reconnect, re-run the three cells above (version pin, Drive mount, repo clone/pull), then re-run this cell with the exact same command. Completed jobs (`results/exp34_*.json` already exists) are skipped automatically — you'll only pay for whatever didn't finish.

In [ ]:
%cd /content/Traceprop/experiments
!python exp34_table2_and_sweep.py --drive_dir /content/drive/MyDrive/traceprop_runs

## Result

The script prints its own summary at the end (also shown above). If you need to re-check after reconnecting without rerunning, this cell reads the same files.

In [ ]:
import json, glob
print('--- Table 2 (head-to-head) ---')
for p in sorted(glob.glob('/content/Traceprop/experiments/results/exp34_table2_*.json')):
    d = json.load(open(p))
    trak = [v for k, v in d.items() if k.startswith('speedup_vs_trak')][0]
    print(f"  {d['model']:<25} flush={d['inline_flush_s']:.3f}s overhead={d['inline_flush_overhead_pct']:.2f}% "
          f"posthoc={d['posthoc_pass_s']:.2f}s LoGRAx={d['speedup_vs_logra_1ckpt']:.1f} TRAKx={trak:.1f}")

print('\n--- Tracked-parameters sweep (batch16 seq64, matches Table 1) ---')
sweep_path = '/content/Traceprop/experiments/results/exp34_sweep.json'
if os.path.exists(sweep_path):
    d = json.load(open(sweep_path))
    for row in d['results']:
        print(f"  {row['label']:<6} layers={row['n_tracked_layers']:<4} grad_dim={row['per_sample_grad_dim']:<8} "
              f"throughput={row['throughput_overhead_pct']:.2f}±{row['throughput_overhead_std']:.2f}%")
else:
    print('  not finished yet -- rerun the cell above')

## Download results locally

Bring these `exp34_*.json` files back to Claude Code to update Table 2 and the tracked-parameters paragraph in `main.tex` — paste the printed summary above, or download the files directly via the Colab file browser (left sidebar → folder icon → `experiments/results/`).